# Chapter 4: Engineer the context layer

Companion notebook to *Agent Harness Engineering* by Praneeth Paikray.

Run every cell in order from a fresh kernel. Sections 1 to 9 use the standard library only. Section 10 is optional and runs P042 through a live planner on the assembled context; it skips itself when no model is configured (set-up is in the Chapter 1 notebook, Section 8).

You will load Chapters 2 and 3, add patient P042 with thirty notes, build the context assembler from the chapter (bundle, derived pending block, size limit, `context` event, `inspect_log` tool), compare it with the naive everything-context, check the invariants at three size limits, work the three exercises, and then watch a live planner run on the bundle.

Every patient, note, protocol, and date is fictional.

## 1. Check the environment

In [ ]:
import sys

assert sys.version_info >= (3, 10), "Use Python 3.10 or newer."
print("Python", ".".join(map(str, sys.version_info[:3])))

## 2. Chapters 2 and 3, condensed

Contracts, reader, grounding, `validate`; then the runtime: `Run`, tools, gate, `step`, `drive`, the stand-in planner, and the misbehaving models. Collapse this cell if you have read those notebooks. One line differs from the Chapter 3 notebook as first published: `drive` now checks the step budget directly (`run.count("proposal") >= run.budget.steps`) instead of assembling a context to read `steps_left`, so that the assembler's `context` event is written exactly once per step.

In [ ]:
from dataclasses import dataclass
from datetime import date

@dataclass(frozen=True)
class Criterion:             # a numeric rule
    id: str
    field: str
    lower: float
    upper: float
    unit: str
    max_age_days: int | None = None   # None: any age
@dataclass(frozen=True)
class Exclusion:             # a rule stated in prose
    id: str
    field: str               # what the model must look for
    window_days: int
@dataclass(frozen=True)
class ProtocolVersion:
    study: str
    version: int
    effective_from: date
    criteria: tuple          # Criterion or Exclusion
@dataclass(frozen=True)
class Evidence:
    id: str
    field: str
    value: float             # 1.0 present, 0.0 absent
    unit: str
    observed: date
@dataclass(frozen=True)
class Note:
    id: str
    written: date
    text: str

AGE = Criterion("age", "age", 18, 65, "years")
PROTOCOLS = [
    ProtocolVersion("T004", 2, date(2026, 1, 15), (AGE,
        Criterion("marker", "marker", 3, 7, "ng/mL"))),
    ProtocolVersion("T004", 3, date(2026, 8, 1), (AGE,
        Criterion("marker", "marker", 4, 8, "ng/mL", 90),
        Exclusion("anticoag", "anticoagulant", 180))),
]
ASSIGNED = {"reviewer-a": {"T004"}, "reviewer-b": {"T009"}}
VOCABULARY = {"age", "marker", "anticoagulant"}

class ContractError(Exception):
    pass

def select_protocol(study, version, on):
    live = [p for p in PROTOCOLS
            if p.study == study and p.effective_from <= on]
    if version == "current" and live:
        return max(live, key=lambda p: p.version)
    for p in live:
        if p.version == version:
            return p
    raise ContractError(f"{study} v{version} not in effect "
                        f"on {on}")

D = date  # shorthand for the fixture tables
RECORDS = {
    "P017": [Evidence("e01", "age", 47, "years", D(2026, 9, 1))],
    "P023": [Evidence("e02", "age", 52, "years", D(2026, 9, 1)),
             Evidence("e03", "marker", 5, "ng/mL", D(2025, 6, 3))],
    "P031": [Evidence("e04", "age", 39, "years", D(2026, 9, 1)),
             Evidence("e05", "marker", 5, "ng/mL", D(2026, 8, 20))],
    "P032": [Evidence("e06", "age", 44, "years", D(2026, 9, 1)),
             Evidence("e07", "marker", 3.5, "ng/mL", D(2026, 8, 22))],
    "P034": [Evidence("e10", "age", 29, "years", D(2026, 9, 1)),
             Evidence("e11", "marker", 5, "nmol/L", D(2026, 8, 25))],
    "P036": [Evidence("e15", "age", 58, "years", D(2026, 9, 1)),
             Evidence("e16", "marker", 6, "ng/mL", D(2026, 8, 28))],
    "P037": [Evidence("e17", "age", 41, "years", D(2026, 9, 1)),
             Evidence("e18", "marker", 5.5, "ng/mL", D(2026, 8, 27))],
    "P038": [Evidence("e19", "age", 36, "years", D(2026, 9, 1)),
             Evidence("e20", "marker", 6.2, "ng/mL", D(2026, 8, 29))],
}
NOTES = {
    "P036": [Note("n01", D(2026, 7, 2),
                  "Atrial fibrillation. Started warfarin 5 mg daily on "
                  "2026-07-02. Follow-up in six weeks.")],
    "P037": [Note("n02", D(2026, 8, 30),
                  "No anticoagulant therapy in the past year, reviewed "
                  "2026-08-30. Continues metformin.")],
    "P038": [Note("n03", D(2026, 8, 29),
                  "Palpitations reported. Discussed starting apixaban at "
                  "the next visit pending cardiology review.")],
}

def liar(context):
    # A model that invents a quotation. Grounding must catch it.
    return {"field": context["field"], "status": "present",
            "observed": "2026-08-01", "quote": "Currently on warfarin"}

def request(patient, version, on=D(2026, 9, 1), by="reviewer-a"):
    return {"patient": patient, "study": "T004", "version": version,
            "on": on, "by": by}

FIXTURES = [
    {"id": "F01", "shape": "all criteria met (v2)",
     "request": request("P031", 2), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "met"}},
    {"id": "F02", "shape": "marker missing",
     "request": request("P017", 2), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "unknown"}},
    {"id": "F03", "shape": "marker stale under v3",
     "request": request("P023", 3), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "unknown", "anticoag": "unknown"}},
    {"id": "F04", "shape": "marker in the wrong unit",
     "request": request("P034", 3), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "unknown", "anticoag": "unknown"}},
    {"id": "F05", "shape": "version not yet in effect",
     "request": request("P031", 3, on=D(2026, 7, 15)),
     "expect": {"status": "refused",
                "reason": "T004 v3 not in effect on 2026-07-15"}},
    {"id": "F06", "shape": "'current' resolves to v3",
     "request": request("P032", "current"),
     "expect": {"status": "prepared", "protocol": "T004 v3"},
     "criteria": {"age": "met", "marker": "not met", "anticoag": "unknown"}},
    {"id": "F07", "shape": "reviewer not assigned to study",
     "request": request("P031", 2, by="reviewer-b"),
     "expect": {"status": "refused",
                "reason": "reviewer-b is not assigned to T004"}},
    {"id": "F08", "shape": "no such patient record",
     "request": request("P999", 2),
     "expect": {"status": "refused", "reason": "no record: P999"}},
    {"id": "F09", "shape": "note states anticoagulant started in window",
     "request": request("P036", 3), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "met", "anticoag": "not met"}},
    {"id": "F10", "shape": "note states absence, with a date",
     "request": request("P037", 3), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "met", "anticoag": "met"}},
    {"id": "F11", "shape": "note mentions the drug but no date",
     "request": request("P038", 3), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "met", "anticoag": "unknown"}},
    {"id": "F12", "shape": "model invents a quotation",
     "request": request("P036", 3), "model": liar,
     "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "met", "anticoag": "unknown"}},
]

def evaluate(criterion, evidence, on):
    items = [e for e in evidence
             if e.field == criterion.field]
    if not items:
        return "unknown", "no value recorded", None
    odd = [e for e in items if e.unit != criterion.unit]
    if odd:
        e = odd[0]
        return "unknown", f"unit is {e.unit}", e.id
    limit = criterion.max_age_days
    if limit is not None:
        fresh = [e for e in items
                 if (on - e.observed).days <= limit]
        if not fresh:
            e = max(items, key=lambda e: e.observed)
            return "unknown", f"stale: {e.observed}", e.id
        items = fresh
    e = max(items, key=lambda e: e.observed)
    inside = criterion.lower <= e.value <= criterion.upper
    return ("met" if inside else "not met",
            f"{e.value} {e.unit} on {e.observed}", e.id)

import re
CUES = ("warfarin", "apixaban", "anticoagulant")

def fake_reader(context):
    # Stand-in for a model reading a note: finds a cue word
    # and returns that sentence with any ISO date in it.
    # A real model reads better; the contract is the same.
    for sentence in context["note"].split(". "):
        low = sentence.lower()
        if any(cue in low for cue in CUES):
            when = re.search(r"\d{4}-\d{2}-\d{2}", sentence)
            absent = low.startswith("no ")
            status = "absent" if absent else "present"
            return {"field": context["field"],
                    "status": status,
                    "observed": when and when.group(),
                    "quote": sentence.strip().rstrip(".")}
    return {"field": context["field"], "status": "none",
            "observed": None, "quote": ""}

def ground(proposal, note):
    # The model proposes. The note and the manifest decide.
    if proposal["field"] not in VOCABULARY:
        return "field not in vocabulary"
    if proposal["status"] == "none":
        return "no statement in note"
    if proposal["quote"] not in note.text:
        return "quote not found in note"
    if proposal["status"] == "unclear":
        return "statement unclear; needs review"
    if not proposal.get("observed"):
        return "statement has no date"
    return None

def read_note(model, note, field):
    proposal = model({"note": note.text, "field": field})
    problem = ground(proposal, note)
    if problem:
        return None, problem
    present = proposal["status"] == "present"
    when = date.fromisoformat(proposal["observed"])
    return Evidence(note.id, field, 1.0 if present else 0.0,
                    "statement", when), None

def evaluate_exclusion(rule, notes, on, model):
    found, why = [], "no statement in notes"
    for note in notes:
        item, problem = read_note(model, note, rule.field)
        if item is None:
            why = f"{note.id}: {problem}"
        else:
            found.append(item)
    recent = [f for f in found
              if (on - f.observed).days <= rule.window_days]
    if any(f.value for f in recent):
        f = next(f for f in recent if f.value)
        return "not met", f"present {f.observed}", f.id
    if recent:
        f = recent[-1]
        return "met", f"absence stated {f.observed}", f.id
    return "unknown", why, None

def validate(req):
    if req["patient"] not in RECORDS:
        raise ContractError(f"no record: {req['patient']}")
    if req["study"] not in ASSIGNED.get(req["by"], set()):
        raise ContractError(f"{req['by']} is not assigned "
                            f"to {req['study']}")
    return select_protocol(req["study"], req["version"],
                           req["on"])

RECORDS["P041"] = [Evidence("e21", "age", 45, "years", D(2026, 9, 1)),
                   Evidence("e22", "marker", 5.5, "ng/mL", D(2026, 8, 26))]
NOTES["P041"] = [Note(f"n{20 + i}", D(2025, 10 + i, 1) if i < 3 else D(2026, i - 2, 1), text)
                 for i, text in enumerate([
    "Physiotherapy review. Knee mobility improving. Continue exercises.",
    "Dermatology follow-up. Eczema settled with emollients.",
    "Annual check. Blood pressure 128/82. No new complaints.",
    "Physiotherapy discharge. Goals met.",
    "Dental referral for wisdom tooth extraction.",
    "Palpitations. ECG shows atrial fibrillation. Started apixaban 5 mg "
    "twice daily on 2026-06-15.",
    "Cardiology review. Rate controlled. Anticoagulant counselling given.",
    "Travel vaccination advice given.",
    "Routine bloods taken. Marker M 5.5 ng/mL.",
])]

from dataclasses import dataclass, field

class BudgetExhausted(Exception):
    pass
@dataclass
class Budget:
    steps: int = 12          # planner turns
    reads: int = 4           # notes the reader may be shown
@dataclass
class Run:
    id: str
    request: dict
    protocol: object
    budget: Budget
    state: str = "running"   # or completed, blocked,
    reason: str = ""         # failed, cancelled
    evidence: list = field(default_factory=list)
    problems: dict = field(default_factory=dict)
    results: dict = field(default_factory=dict)
    events: list = field(default_factory=list)
    reads: int = 0
    cancel: bool = False

    def record(self, kind, **data):
        self.events.append({"seq": len(self.events),
                            "kind": kind, **data})

    def count(self, kind):
        return sum(e["kind"] == kind for e in self.events)

def start(req, budget=None):
    protocol = validate(req)      # raises ContractError
    run = Run(f"run-{req['patient']}-{req['on']}", req,
              protocol, budget or Budget())
    run.evidence = list(RECORDS[req["patient"]])
    run.record("request", patient=req["patient"],
               protocol=protocol.version)
    return run

def judge_exclusion(rule, evidence, problems, on):
    found = [e for e in evidence if e.field == rule.field]
    recent = [f for f in found
              if (on - f.observed).days <= rule.window_days]
    if any(f.value for f in recent):
        f = next(f for f in recent if f.value)
        return "not met", f"present {f.observed}", f.id
    if recent:
        f = recent[-1]
        return "met", f"absence stated {f.observed}", f.id
    if problems:
        nid, why = next(iter(problems.items()))
        return "unknown", f"{nid}: {why}", None
    return "unknown", "no grounded statement", None

def t_search(run, terms):
    notes = NOTES.get(run.request["patient"], [])
    words = [t.lower() for t in terms]
    hits = [n.id for n in notes
            if any(w in n.text.lower() for w in words)]
    return {"notes": hits}
def t_read(run, note_id, reader):
    if run.reads >= run.budget.reads:
        raise BudgetExhausted("read budget spent")
    run.reads += 1
    note = next(n for n in NOTES[run.request["patient"]]
                if n.id == note_id)
    item, problem = read_note(reader, note, "anticoagulant")
    if item is None:
        run.problems[note_id] = problem
        return {"note": note_id, "problem": problem}
    run.evidence.append(item)
    return {"note": note_id, "present": bool(item.value),
            "observed": str(item.observed)}
def t_evaluate(run, rule_id):
    rule = next(r for r in run.protocol.criteria
                if r.id == rule_id)
    on = run.request["on"]
    if isinstance(rule, Exclusion):
        result = judge_exclusion(rule, run.evidence,
                                 run.problems, on)
    else:
        result = evaluate(rule, run.evidence, on)
    run.results[rule_id] = result
    return {"rule": rule_id, "status": result[0],
            "reason": result[1]}

TOOLS = {"search_notes": t_search, "read_note": t_read,
         "evaluate_rule": t_evaluate}
ARGS = {"search_notes": {"terms"}, "read_note": {"note_id"},
        "evaluate_rule": {"rule_id"}, "finish": set()}

def gate(run, proposal):
    tool = proposal.get("tool")
    args = proposal.get("args", {})
    if tool not in ARGS:
        return f"tool not allowed: {tool}"
    if set(args) != ARGS[tool]:
        return f"bad arguments for {tool}: {sorted(args)}"
    rules = [r.id for r in run.protocol.criteria]
    if tool == "evaluate_rule":
        if args["rule_id"] not in rules:
            return f"no such rule: {args['rule_id']}"
    return None

def context_for(run):
    return {"task": run.request,
            "rules": [r.id for r in run.protocol.criteria],
            "results": {k: v[0]
                        for k, v in run.results.items()},
            "observations": [e for e in run.events
                             if e["kind"] == "observation"],
            "steps_left": (run.budget.steps
                           - run.count("proposal"))}

def step(run, planner, reader):
    proposal = planner(context_for(run))
    run.record("proposal", **proposal)
    refusal = gate(run, proposal)
    if refusal:
        run.record("gate", refused=refusal)
        run.record("observation", error=refusal)
        return
    tool, args = proposal["tool"], proposal.get("args", {})
    if tool == "finish":
        missing = [r.id for r in run.protocol.criteria
                   if r.id not in run.results]
        if missing:
            run.record("observation",
                       error=f"cannot finish: {missing}")
            return
        run.state = "completed"
        run.reason = "every rule has a result"
        return
    if tool == "read_note":
        args = {**args, "reader": reader}
    observation = TOOLS[tool](run, **args)
    run.record("observation", tool=tool, **observation)

def drive(run, planner, reader=fake_reader):
    while run.state == "running":
        if run.cancel:
            run.state = "cancelled"
            run.reason = "asked to stop"
        elif run.count("proposal") >= run.budget.steps:
            run.state = "blocked"
            run.reason = "step budget spent"
        else:
            try:
                step(run, planner, reader)
            except BudgetExhausted as why:
                run.state, run.reason = "blocked", str(why)
            except Exception as why:      # a tool failed
                run.state = "failed"
                run.reason = f"{type(why).__name__}: {why}"
    run.record("stop", state=run.state, reason=run.reason)
    return run

def run_task(req, planner, reader=fake_reader, budget=None):
    try:
        run = start(req, budget)
    except ContractError as why:
        return {"status": "refused", "reason": str(why)}
    return drive(run, planner, reader)

SEARCH_TERMS = ["anticoagulant", "warfarin", "apixaban"]

def fake_planner(context):
    # Stand-in for a model choosing the next step. It takes
    # the rules in order and searches before it reads.
    obs = context["observations"]
    for rule_id in context["rules"]:
        if rule_id in context["results"]:
            continue
        if rule_id != "anticoag":
            return {"tool": "evaluate_rule",
                    "args": {"rule_id": rule_id}}
        searches = [e for e in obs
                    if e.get("tool") == "search_notes"]
        if not searches:
            return {"tool": "search_notes",
                    "args": {"terms": SEARCH_TERMS}}
        read = {e["note"] for e in obs
                if e.get("tool") == "read_note"}
        todo = [n for n in searches[-1]["notes"]
                if n not in read]
        if todo:
            return {"tool": "read_note",
                    "args": {"note_id": todo[0]}}
        return {"tool": "evaluate_rule",
                "args": {"rule_id": rule_id}}
    return {"tool": "finish"}

def loopy(context):      # never stops searching
    return {"tool": "search_notes",
            "args": {"terms": ["anything"]}}

def eager(context):      # declares victory at once
    return {"tool": "finish"}
def rogue(context):      # tries a tool it was never given
    if not context["observations"]:
        return {"tool": "write_record",
                "args": {"marker": 5}}
    return fake_planner(context)

def broken_reader(context):
    raise RuntimeError("model endpoint timed out")

## 3. P042: thirty notes, two that matter

A year of routine visits with one cardiology episode. Notes n58 and n61 mention anticoagulants; n61 has no date.

In [ ]:
from datetime import timedelta

RECORDS["P042"] = [Evidence("e23", "age", 50, "years", D(2026, 9, 1)),
                   Evidence("e24", "marker", 6.1, "ng/mL", D(2026, 8, 24))]
ROUTINE = [
    "Physiotherapy session. Shoulder range improving. Home exercises reviewed.",
    "Dermatology review. Psoriasis plaques reduced on current topical treatment.",
    "Routine check. Blood pressure 124/80. Weight stable. No new complaints.",
    "Dietitian consultation. Mediterranean pattern discussed. Follow-up in three months.",
    "Optometry. Mild presbyopia. New prescription issued.",
    "Travel clinic. Hepatitis A booster given. Malaria advice for the region discussed.",
    "Podiatry. Callus debrided. Footwear advice given.",
    "Dental hygiene appointment. Scaling completed.",
]
CARDIAC = {
    18: "Palpitations reported. ECG shows atrial fibrillation. Started apixaban "
        "5 mg twice daily on 2026-04-10.",
    21: "Cardiology follow-up. Rate controlled. Anticoagulant counselling "
        "reinforced; bleeding precautions discussed.",
}
NOTES["P042"] = [Note(f"n{40 + i}", D(2025, 9, 1) + timedelta(days=12 * i),
                      CARDIAC.get(i, ROUTINE[i % len(ROUTINE)]))
                 for i in range(30)]

print(len(NOTES["P042"]), "notes;", [n.id for n in NOTES["P042"] if any(c in n.text.lower() for c in CUES)], "mention anticoagulants")

## 4. Listing 4.1: the bundle and the pending block

In [ ]:
import json
from dataclasses import asdict

@dataclass
class Context:
    task: dict
    rules: list       # selected version only, sourced
    results: dict
    pending: dict     # derived from the whole log
    recent: list      # the last few observations, verbatim
    archive: dict     # what was left out, how to reach it
    steps_left: int

def describe(rule, protocol):
    d = asdict(rule)
    d["source"] = f"{protocol.study} v{protocol.version}"
    return d
def observations(run):
    return [e for e in run.events
            if e["kind"] == "observation"]
def pending_for(run):
    obs = observations(run)
    searches = [e for e in obs
                if e.get("tool") == "search_notes"]
    hits = {n for e in searches for n in e["notes"]}
    read = {e["note"] for e in obs
            if e.get("tool") == "read_note"}
    todo = [r.id for r in run.protocol.criteria
            if r.id not in run.results]
    return {"searched": bool(searches),
            "unread_hits": sorted(hits - read),
            "rules_without_result": todo,
            "problems": dict(run.problems),
            "reads_left": run.budget.reads - run.reads}

## 5. Listing 4.2: the assembler

`assemble` replaces Chapter 3's `context_for`. The recent window shrinks oldest-first until the bundle fits the limit; the size and what was left out are written to the log as a `context` event before the planner sees the bundle.

In [ ]:
def size(ctx):
    return len(json.dumps(asdict(ctx), default=str))

def assemble(run, recent=4, limit=2400):
    obs = observations(run)
    ctx = Context(
        task={"patient": run.request["patient"],
              "on": str(run.request["on"])},
        rules=[describe(r, run.protocol)
               for r in run.protocol.criteria],
        results={k: v[0] for k, v in run.results.items()},
        pending=pending_for(run),
        recent=obs[-recent:],
        archive={"events": len(run.events),
                 "observations": len(obs),
                 "how": "inspect_log(kind, last)"},
        steps_left=run.budget.steps - run.count("proposal"))
    while size(ctx) > limit and len(ctx.recent) > 1:
        ctx.recent = ctx.recent[1:]     # oldest goes first
    run.record("context", chars=size(ctx),
               recent=len(ctx.recent),
               archived=len(obs) - len(ctx.recent))
    return asdict(ctx)

context_for = assemble      # Chapter 3's step() now uses it

## 6. Listing 4.3: a tool into the archive, and a planner that reads the pending block

In [ ]:
def t_inspect(run, kind, last):
    picked = [e for e in run.events if e["kind"] == kind]
    return {"events": [{k: v for k, v in e.items()
                        if k != "kind"}
                       for e in picked[-last:]]}
TOOLS["inspect_log"] = t_inspect
ARGS["inspect_log"] = {"kind", "last"}
def planner_v2(context):
    # Works from the derived pending block, not history.
    p = context["pending"]
    if p["unread_hits"] and p["reads_left"] > 0:
        return {"tool": "read_note",
                "args": {"note_id": p["unread_hits"][0]}}
    for rule_id in p["rules_without_result"]:
        if rule_id == "anticoag" and not p["searched"]:
            return {"tool": "search_notes",
                    "args": {"terms": SEARCH_TERMS}}
        return {"tool": "evaluate_rule",
                "args": {"rule_id": rule_id}}
    return {"tool": "finish"}

## 7. Listing 4.4: the anti-pattern, the invariants, and the exit test

Expected: the run completes reading 2 of 30 notes; the naive bundle is about 6,500 characters; the assembled bundle is about 1,400 at a 2,400 limit, 1,140 at 1,200, and 830 at 700 with one observation left in the window. The invariants hold at every size.

In [ ]:
def naive_context(run):
    # Everything, for completeness. The anti-pattern.
    notes = NOTES.get(run.request["patient"], [])
    return {"protocols": [asdict(p) for p in PROTOCOLS],
            "record": [asdict(e) for e in run.evidence],
            "notes": [asdict(n) for n in notes],
            "events": run.events}
def invariants(run, ctx):
    want = pending_for(run)
    version = f"T004 v{run.protocol.version}"
    assert all(r["source"] == version for r in ctx["rules"])
    assert "[3, 7]" not in json.dumps(ctx, default=str)
    got = ctx["pending"]
    assert got["unread_hits"] == want["unread_hits"]
    assert (got["rules_without_result"]
            == want["rules_without_result"])
    return "hold"
run = run_task(request("P042", 3), planner_v2)
print(run.state, "| notes read:", run.reads,
      "of", len(NOTES["P042"]))
naive = len(json.dumps(naive_context(run), default=str))
print(f"naive context: {naive} chars")
for limit in (2400, 1200, 700):
    ctx = assemble(run, recent=8, limit=limit)
    print(f"limit {limit}: {size(Context(**ctx))} chars,"
          f" recent {len(ctx['recent'])}, invariants",
          invariants(run, ctx))

The checks below are what the chapter asserts.

In [ ]:
assert run.state == "completed" and run.reads == 2
fresh = run_task(request("P042", 3), planner_v2)      # Listing 4.4 called assemble three extra times on `run`
kinds = [e["kind"] for e in fresh.events]
assert kinds.count("context") == kinds.count("proposal"), "one context event per step"
ctx = assemble(run, recent=8, limit=700)
assert len(ctx["recent"]) == 1 and invariants(run, ctx) == "hold"
assert all(r["source"] == "T004 v3" for r in ctx["rules"])
assert naive > 4 * size(Context(**ctx))
print("Chapter 4 behaves as described. Context events:", kinds.count("context"))
print("The log now records what the planner was shown at each step:")
for e in fresh.events:
    if e["kind"] == "context": print("  ", e)

## 8. Exercises

### Exercise 4.1: the old planner on the new bundle

Run Chapter 3's `fake_planner` against the assembler and read the log.

In [ ]:
old = run_task(request("P042", 3), fake_planner)
print(old.state, "|", old.reason)
for e in old.events[-3:]: print(e)

**Solution 4.1.** It fails on the first step with `KeyError: 'observations'`, recorded as a `failed` run with the exception in the reason: the runtime treated the planner's crash as a dependency breaking, which is what it was. Is the defect in the planner or the assembler? In the planner. A planner is entitled to assume the *shape* of its context (the six fields of Listing 4.1) and nothing about its *length*; a planner that needs the whole history has made an assumption the harness never promised. `planner_v2` makes only the promised assumption: it reads `pending`, which the assembler guarantees is complete however small the window. The general rule: anything a planner must not forget belongs in a derived field, never in the window.

### Exercise 4.2: a fifth invariant

No note text in the bundle: only identifiers and grounded statements.

In [ ]:
def no_note_text(run, ctx):
    text = json.dumps(ctx, default=str)
    for n in NOTES.get(run.request["patient"], []):
        for sentence in n.text.split(". "):
            if len(sentence) > 20 and sentence.strip(".") in text:
                return f"note text leaked from {n.id}: {sentence[:40]!r}"
    return "hold"

ctx = assemble(run)
print("as built:", no_note_text(run, ctx))

# Break it: a read tool that returns the sentence it quoted.
_t_read = t_read
def t_read(run, note_id, reader):
    out = _t_read(run, note_id, reader)
    note = next(n for n in NOTES[run.request["patient"]] if n.id == note_id)
    prop = reader({"note": note.text, "field": "anticoagulant"})
    return {**out, "quote": prop.get("quote", "")}
TOOLS["read_note"] = t_read
leaky = run_task(request("P042", 3), planner_v2)
print("leaky:   ", no_note_text(leaky, assemble(leaky)))
t_read = _t_read; TOOLS["read_note"] = _t_read

**Solution 4.2.** As built, the invariant holds: `read_note` returns `present`/`observed`/`problem`, never the sentence. The leaky version puts the quoted sentence into the observation, and from there into `recent`, and the invariant catches it. What it violates is Chapter 2's line between context and evidence (Section 1.3 and Listing 2.5): the quotation belongs in the *packet*, cited by note id, for the reviewer; it does not belong in the planner's context, where the model would be re-reading prose the harness has already judged. A reviewer would see nothing different in the packet, which is exactly the problem: the leak is invisible downstream and only a check on the bundle finds it.

### Exercise 4.3: the note lexical search cannot find

A note that describes anticoagulant therapy without any of the three search terms.

In [ ]:
NOTES["P042"].append(Note("n70", D(2026, 8, 28),
    "Blood thinner started after the arrhythmia clinic visit on 2026-08-20; "
    "patient counselled on bleeding risk."))
# Remove the two cue-word notes so the new one is the only evidence.
hidden = [n for n in NOTES["P042"] if n.id in ("n58", "n61")]
NOTES["P042"] = [n for n in NOTES["P042"] if n.id not in ("n58", "n61")]
r = run_task(request("P042", 3), planner_v2)
print(r.results["anticoag"])
NOTES["P042"] = [n for n in NOTES["P042"] if n.id != "n70"] + hidden

**Solution 4.3.** Lexical search returns nothing, the planner evaluates the exclusion with no statements, and the result is `unknown: no grounded statement`. That is the honest answer for this harness, and it is wrong about the patient.

The fixture that would justify vector search: P042 with note n70 only, expected `anticoag: not met, present 2026-08-20, [n70]`. Vector search over the notes would rank n70 for a query like "anticoagulant therapy" because "blood thinner" is close in embedding space. But the retrieval step only returns an identifier. `read_note` still has to run, the reader has to return a statement with a verbatim quotation and a date, and `ground` has to accept it. Vector search widens what can be *found*; it changes nothing about what can be *admitted*. If you add it, add a ceiling on results (the read budget already provides one) and keep the grounding check exactly where it is. The book will add it when a fixture like this one is in the suite and failing.

## 9. Sketch: a bounded recursive read over the archive

Section 4.6 describes recursive language models: a model that inspects a long input, slices it, and calls itself on the slices with a fresh budget. The harness already does a small version of this with one note per reader call. Here is the shape of the general one, with the stand-in reader standing in for the sub-call, so you can see what "bounded" means in code.

In [ ]:
def recursive_scan(run, field, reader, slice_size=3, max_slices=4):
    # Read the archive in slices, each under its own budget; return grounded findings.
    notes = NOTES.get(run.request["patient"], [])
    findings, scanned = [], 0
    for i in range(0, len(notes), slice_size):
        if scanned >= max_slices:
            break
        scanned += 1
        for note in notes[i:i + slice_size]:
            item, problem = read_note(reader, note, field)
            if item is not None:
                findings.append((note.id, item.value, str(item.observed)))
    return {"slices_scanned": scanned, "notes_seen": min(len(notes), scanned * slice_size),
            "findings": findings}

print(recursive_scan(run, "anticoagulant", fake_reader, slice_size=5, max_slices=6))

Every sub-call sees at most `slice_size` notes; the whole scan sees at most `max_slices` slices; findings come back as grounded statements, never text. That is the entire discipline. Whether it is worth a tool of its own is a question for a fixture that lexical search fails.

## 10. Optional: a live planner on the assembled bundle

Configure a model as in the Chapter 1 notebook. `live_planner_v2` receives exactly the six-field bundle of Listing 4.1 and the tool list, including `inspect_log`. Compare its log with `planner_v2`'s: how many steps, how many reads, whether it ever reaches into the archive, and how large each `context` event was.

In [ ]:
import os, urllib.request, urllib.error, json, datetime

try:
    from databricks.sdk import WorkspaceClient
    _w = WorkspaceClient()
    os.environ.setdefault("DATABRICKS_HOST", _w.config.host)
    _auth = _w.config.authenticate().get("Authorization", "")
    if _auth.startswith("Bearer "):
        os.environ.setdefault("DATABRICKS_TOKEN", _auth.split(" ", 1)[1])
except Exception:
    pass
LIVE = bool(os.environ.get("DATABRICKS_HOST") and os.environ.get("DATABRICKS_TOKEN"))
ENDPOINT = os.environ.get("MODEL_ENDPOINT", "databricks-gpt-5-6-luna")

def chat_json(prompt, fallback):
    host = os.environ["DATABRICKS_HOST"].rstrip("/")
    body = json.dumps({"model": ENDPOINT, "messages": [{"role": "user", "content": prompt}]}).encode()
    req = urllib.request.Request(host + "/serving-endpoints/chat/completions", body,
                                 {"Content-Type": "application/json",
                                  "Authorization": "Bearer " + os.environ["DATABRICKS_TOKEN"]})
    with urllib.request.urlopen(req, timeout=60) as reply:
        text = json.load(reply)["choices"][0]["message"]["content"]
    try:
        return json.loads(text.strip().strip("`").removeprefix("json").strip())
    except ValueError:
        return fallback(text)

def live_planner_v2(context):
    prompt = (
        "You are the planner inside a clinical-trial screening harness. Choose the single next step. "
        "Reply with JSON only: {\"tool\": <name>, \"args\": {...}}.\n"
        "Tools and exact argument names:\n"
        "  search_notes {\"terms\": [words]} -> ids of notes containing any term\n"
        "  read_note {\"note_id\": id} -> grounded statement about anticoagulant use, or a problem\n"
        "  evaluate_rule {\"rule_id\": id} -> status for one rule from evidence gathered so far\n"
        "  inspect_log {\"kind\": one of request|proposal|gate|observation|context, \"last\": n} -> archive slice\n"
        "  finish {} -> completion check; refused if any rule lacks a result\n"
        "Your context (the bundle). 'pending' is authoritative about what is still owed; 'recent' is only a window.\n"
        + json.dumps(context, default=str) + "\n"
        "Only 'anticoag' depends on notes. Read unread hits before evaluating it. Do not re-read a note."
    )
    return chat_json(prompt, lambda text: {"tool": "unparseable", "args": {"text": text[:80]}})

print("Live model available:", LIVE, "| endpoint:", ENDPOINT if LIVE else "(none)")

In [ ]:
if LIVE:
    print(f"{datetime.date.today()}  endpoint={ENDPOINT}")
    live = run_task(request("P042", 3), live_planner_v2, budget=Budget(steps=12, reads=4))
    for e in live.events:
        print(e)
    print()
    print(live.state, "|", live.reason, "| steps:", live.count("proposal"), "| reads:", live.reads,
          "| gate refusals:", live.count("gate"), "| archive lookups:",
          sum(1 for e in live.events if e.get("tool") == "inspect_log"))
    sizes = [e["chars"] for e in live.events if e["kind"] == "context"]
    print("context sizes per step:", sizes)
else:
    print("Skipped: no live model configured.")

### What to look for

The `context` events give you the size of what the model saw at every step; with the stand-in the numbers are in Table 4.3. A live planner may use `inspect_log`, which the stand-in never does, and you can see whether it did so for a reason (a forgotten search) or out of curiosity (a wasted step). Anything it proposes that the gate refuses is in the log. Print endpoint and date with anything you quote.

## 11. Done

Context is assembled from the log each step under a size limit, the pending block cannot be compacted away, the archive is one tool call away, and the log records what the model was shown. Chapter 5 turns to the tools: schemas, preconditions, errors, and the line between reading, proposing, and committing.

In [ ]:
print("Chapter 4 complete.")

## References

- [Anthropic, Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)
- [Anthropic, Scaling Managed Agents](https://www.anthropic.com/engineering/managed-agents)
- [Zhang, Kraska, Khattab, Recursive Language Models](https://arxiv.org/abs/2512.24601)
- [Anthropic, Code execution with MCP](https://www.anthropic.com/engineering/code-execution-with-mcp)
- [Agent Skills specification](https://agentskills.io/specification)
- [Anthropic, Introducing Contextual Retrieval](https://www.anthropic.com/engineering/contextual-retrieval)

The assembler, notes, and exercises are original companion material.